In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from llama_cpp import Llama
import json

app = FastAPI()

# 使用 r"" 來避免 Windows 路徑的反斜線跳脫字元問題
# n_gpu_layers=-1 將運算全數交由顯卡處理，發揮硬體最大效能
llm = Llama(
    model_path=r"C:\Users\user\Desktop\新增資料夾 (3)\Mistral-Nemo-Instruct-2407-Q4_K_M.gguf",
    n_gpu_layers=-1, 
    n_ctx=2048       
)

class WordRequest(BaseModel):
    word: str

@app.post("/api/learning/generate-sentence")
async def generate_hakka_sentence(request: WordRequest):
    target_word = request.word
    
    # 轉換為 Mistral-Nemo 專屬的 [INST] 指令格式
    # 同樣利用 Few-Shot 範例來強制約束 JSON 輸出
    prompt = f"""[INST] 你是一個專業的台灣客語教師。請根據輸入的中文單字，生成一句適合國小生學習的生活化客語例句（長度在 10 個字以內）。
請務必只輸出合法的 JSON 格式，包含 "hakka_sentence" 與 "chinese_translation" 兩個鍵值，絕對不要輸出任何其他說明文字。

輸入：椅子 [/INST]
{{"hakka_sentence": "這張椅子當好坐。", "chinese_translation": "這張椅子很好坐。"}}

[INST] 輸入：吃飯 [/INST]
{{"hakka_sentence": "大家來食飯囉。", "chinese_translation": "大家來吃飯囉。"}}

[INST] 輸入：{target_word} [/INST]
"""
    
    try:
        # temperature 調低至 0.2，讓模型輸出更穩定、減少幻覺
        response = llm(
            prompt,
            max_tokens=100,
            temperature=0.2,
            echo=False
        )
        
        result_text = response["choices"][0]["text"].strip()
        parsed_result = json.loads(result_text)
        return parsed_result
        
    except json.JSONDecodeError:
        raise HTTPException(status_code=500, detail="模型未輸出正確的 JSON 格式")
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"本地端推論發生錯誤: {str(e)}")